## Vector Index
- Finding semantically similar vector is stright forward
- However, brute-force method is computationly costly
    - n * (n-1) process
- **Vector index** designed to organize the high-dimentional vectors for fast nearest neighbors search
    - insted of treating dataset as falt, the index treats the data that refelcts the geometrz of the vector space
    - clustering similar vectors tougther or linking them through proximity-based graphs
    - Hence, the neighbouring search can prune a large portion of the dataset early in the search process.

## Hierarchical Navigable Small World (HNSW)
- fast, scalabel, graph-based vector index
- Approximate nearest neighbor search in High-dimentional space
- the only indexing method supported by chroma db

### Working
- is a multi-layered graph
    - the upper layers contain a sparse overview of the data for fast navigation
    - the bottom lazer holds all vectors for detailed search
- each vector connected to a few nearby neighbors (small world)

### Search process
- algoritham starts ar the top layer and move towards the query vector as it desents
- as it decents (refinign the search at each level), the sturcture allows it to skip most of vectors

### Advantages of HNSW
- Fast: aviod scanning the entire dataset
- Accurate: deliver near exact result
- Scalable: Handle millions to billions of vectors
- Versatile: works with various similarity metrics

## Setting up HNSW in Chroma db
- during collection creation

In [3]:
# Setup
import chromadb
from chromadb.utils import embedding_functions

ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# collection creation
client = chromadb.Client()
collection = client.create_collection(
    name="my_collection_name"
    , metadata = {
        "topic": "query_testing"
    }
    , configuration={
        "hnsw": {
            "space": "cosine",
            "ef_search": 100,
            "ef_construction": 100,
            "max_neighbors": 16
        }
        , "embedding_function": ef
    }
)


### key configuration parameters are
- **space**: select the distance measures, options include
    - l2: Euclidean distance (default)
    - ip: inner dot product
    - cosine
- **ef_search**
    - size of the candidate list used to search for nearest neighbors, when a search is performed
    - defualt value is 100
    - High value improves accuracz and recall
    - slower performance and high computational cost
- **ef_construction**
    - size of candidate list used to selct neighbours when a node is inserted for index construction
    - default value is 100
    - Higher value improve the qualitz of the index and accuracy
    - slow performance and inreased memory usage
- **max_neighbors**
    - maximum number of consturctions each node can have during construction
    - default value is 16
    - Higher value leads to denser graphs that perform better searches
    - However, high memory usage and construction time

- can categorize the performance-based parameters into two types
    - **ef_search** controls the bredth of the search at the query time
        - recall vs query speed
    - **ef_construction** and **max_neighbors** affect the quality of the build index
        - Higher-qualitz, deser index (with **ef_construction** and **max_neighbors**) provides a better graph leads to a better search
        - the qualitz comes at the cost of significantlz longer index build times

## Perfroming similarity search in Chroma DB

### Add data

In [4]:
collection.add(
    documents=[
        "Giant pandas are a bear species that lives in mountainous areas.",
        "A pandas DataFrame stores two-dimensional, tabular data",
        "I think everyone agrees that pandas are some of the cutest animals on the planet",
        "A direct comparison between pandas and polars indicates that polars is a more efficient library than pandas.",
    ],
    metadatas=[
        {"topic": "animals"},
        {"topic": "data analysis"},
        {"topic": "animals"},
        {"topic": "data analysis"},
    ],
    ids=["id1", "id2", "id3", "id4"]
)

- all the document conatins the word pandas
- However, id1 and id3 the animal and id2 and id4 refers to the library

## Querying in Chroma db
- query based on cats

In [6]:
collection.query(
    query_texts=["cats"],
    n_results=10
)

{'ids': [['id3', 'id1', 'id2', 'id4']],
 'embeddings': None,
 'documents': [['I think everyone agrees that pandas are some of the cutest animals on the planet',
   'Giant pandas are a bear species that lives in mountainous areas.',
   'A pandas DataFrame stores two-dimensional, tabular data',
   'A direct comparison between pandas and polars indicates that polars is a more efficient library than pandas.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'topic': 'animals'},
   {'topic': 'animals'},
   {'topic': 'data analysis'},
   {'topic': 'data analysis'}]],
 'distances': [[0.7380144596099854,
   0.8351748585700989,
   0.8634341955184937,
   0.9299634099006653]]}

- Text with animal pandas has a less distance compared to the word pandas used as a library

In [7]:
collection.query(
    query_texts=["polar bear"],
    n_results=1,
    where={'topic': 'animals'}
)

{'ids': [['id1']],
 'embeddings': None,
 'documents': [['Giant pandas are a bear species that lives in mountainous areas.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'topic': 'animals'}]],
 'distances': [[0.7096825838088989]]}

- we can use a metadata query to specify that we are looking for animal
- give more context in the question to make it clear that we are looking for an animal

In [8]:
# metadata filter that says to omit libraries
collection.query(
    query_texts=["polar bear"],
    n_results=1,
    where_document={'$not_contains': 'library'}
)

{'ids': [['id1']],
 'embeddings': None,
 'documents': [['Giant pandas are a bear species that lives in mountainous areas.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'topic': 'animals'}]],
 'distances': [[0.7096825838088989]]}

- we can combine both,
    - metadata filter
    - text filter

In [9]:
collection.query(
    query_texts=["polar bear"],
    n_results=1,
    where={"topic": "animals"},
    where_document={"$not_contains": "library"}
)

{'ids': [['id1']],
 'embeddings': None,
 'documents': [['Giant pandas are a bear species that lives in mountainous areas.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'topic': 'animals'}]],
 'distances': [[0.7096825838088989]]}